# 🔬 Phase 5: Forward-Forward (FF) Algorithm vs Backpropagation
### 〜 誤差逆伝播の完全排除とアクティベーションメモリゼロへの挑戦 〜

本ノートブックでは、Geoffrey Hinton が 2022 年に提唱した **Forward-Forward Algorithm** を 1.58-bit Bit-SSM に適用し、
従来の **誤差逆伝播法（Backpropagation）** と比較検証します。

#### 🌟 主な検証項目:
1. **アクティベーションメモリ:** 逆伝播（全層保持） vs Forward-Forward（1層のみ保持）のメモリ消費比較
2. **Goodness (良さ度) 分離度:** 正例文脈と負例文脈（破損データ）を各層が局所的に識別できるかの評価
3. **Zero-Backward テキスト生成:** グローバルな勾配計算を一切行わずに獲得した表現でのテキスト自己回帰生成

In [ ]:
import time
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Running on: {device}")

## 1. 1.58-bit 量子化 & Forward-Forward SSM ブロックの定義 (v2 改良版)

In [ ]:
class Quantize158(torch.autograd.Function):
    @staticmethod
    def forward(ctx, weight):
        gamma = weight.abs().mean().clamp(min=1e-5)
        w_scaled = weight / gamma
        w_ternary = torch.clamp(torch.round(w_scaled), -1.0, 1.0)
        return w_ternary * gamma

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output

class BitLinear158(nn.Linear):
    def forward(self, x):
        w_q = Quantize158.apply(self.weight)
        return F.linear(x, w_q, self.bias)

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x * norm * self.weight

class ForwardForwardSSMBlock(nn.Module):
    def __init__(self, d_model: int, d_state: int = 16, vocab_size: int = 500, lr: float = 3e-3):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.vocab_size = vocab_size

        self.norm1 = RMSNorm(d_model)
        self.in_proj = BitLinear158(d_model, 2 * d_model)
        self.conv1d = nn.Conv1d(d_model, d_model, kernel_size=4, padding=3, groups=d_model)
        self.b_proj = BitLinear158(d_model, d_state)
        self.c_proj = BitLinear158(d_model, d_state)
        self.decay_param = nn.Parameter(torch.tensor([-1.0] * d_state))
        self.out_proj = BitLinear158(d_model, d_model)

        self.norm2 = RMSNorm(d_model)
        self.ffn_in = BitLinear158(d_model, d_model * 4)
        self.ffn_out = BitLinear158(d_model * 2, d_model)

        self.local_head = nn.Linear(d_model, vocab_size, bias=False)
        self.optimizer = torch.optim.AdamW(self.parameters(), lr=lr, weight_decay=0.01)

    def _ssm_step(self, x: torch.Tensor) -> torch.Tensor:
        B, L, D = x.shape
        nx = self.norm1(x)
        proj = self.in_proj(nx)
        u, gate = proj.chunk(2, dim=-1)

        u_conv = self.conv1d(u.transpose(1, 2))[:, :, :L].transpose(1, 2)
        u_conv = F.silu(u_conv)

        B_t = self.b_proj(u_conv)
        C_t = self.c_proj(u_conv)
        decay = torch.sigmoid(self.decay_param)

        h = torch.zeros(B, self.d_state, device=x.device, dtype=x.dtype)
        y_list = []
        u_scalar = u_conv.mean(dim=-1, keepdim=True)
        for t in range(L):
            h = decay * h + B_t[:, t, :] * u_scalar[:, t, :]
            y_t = (C_t[:, t, :] * h).sum(dim=-1, keepdim=True)
            y_list.append(y_t)
        y_ssm = torch.cat(y_list, dim=-1).unsqueeze(-1).expand(-1, -1, D)

        ssm_out = self.out_proj((u_conv + y_ssm) * F.silu(gate))
        x = x + ssm_out

        nx2 = self.norm2(x)
        ffn_p = self.ffn_in(nx2)
        f1, f2 = ffn_p.chunk(2, dim=-1)
        ffn_act = F.silu(f1) * f2
        return x + self.ffn_out(ffn_act)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self._ssm_step(x)

    def train_step_ff(self, h_pos: torch.Tensor, h_neg: torch.Tensor, target_ids: torch.Tensor) -> dict:
        self.optimizer.zero_grad()
        out_pos = self._ssm_step(h_pos)
        out_neg = self._ssm_step(h_neg)

        g_pos = (out_pos ** 2).mean(dim=-1)
        g_neg = (out_neg ** 2).mean(dim=-1)
        loss_ff = F.softplus(g_neg - g_pos + 1.0).mean()

        logits = self.local_head(out_pos)
        loss_lm = F.cross_entropy(logits.view(-1, self.vocab_size), target_ids.view(-1))

        total_loss = loss_lm + 0.1 * loss_ff
        total_loss.backward()
        self.optimizer.step()

        with torch.no_grad():
            next_h_pos = out_pos / (out_pos.norm(2, dim=-1, keepdim=True) + 1e-6)
            next_h_neg = out_neg / (out_neg.norm(2, dim=-1, keepdim=True) + 1e-6)

        return {
            'loss_ff': float(loss_ff.item()),
            'loss_lm': float(loss_lm.item()),
            'g_pos': float(g_pos.mean().item()),
            'g_neg': float(g_neg.mean().item()),
            'next_h_pos': next_h_pos.detach(),
            'next_h_neg': next_h_neg.detach()
        }

## 2. Forward-Forward モデルと正例/負例データローダー

In [ ]:
class ForwardForwardBitSSM(nn.Module):
    def __init__(self, vocab_size: int = 500, d_model: int = 64, n_layers: int = 3, d_state: int = 16, lr: float = 3e-3):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.n_layers = n_layers
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.emb_head = nn.Linear(d_model, vocab_size, bias=False)
        self.emb_optimizer = torch.optim.AdamW(list(self.embedding.parameters()) + list(self.emb_head.parameters()), lr=lr)
        self.blocks = nn.ModuleList([
            ForwardForwardSSMBlock(d_model=d_model, d_state=d_state, vocab_size=vocab_size, lr=lr)
            for _ in range(n_layers)
        ])

    def train_step(self, pos_tokens: torch.Tensor, neg_tokens: torch.Tensor, target_ids: torch.Tensor) -> list:
        self.emb_optimizer.zero_grad()
        h_pos = self.embedding(pos_tokens)
        emb_logits = self.emb_head(h_pos)
        emb_loss = F.cross_entropy(emb_logits.view(-1, self.vocab_size), target_ids.view(-1))
        emb_loss.backward()
        self.emb_optimizer.step()

        with torch.no_grad():
            cur_pos = (h_pos / (h_pos.norm(2, dim=-1, keepdim=True) + 1e-6)).detach()
            h_neg = self.embedding(neg_tokens)
            cur_neg = (h_neg / (h_neg.norm(2, dim=-1, keepdim=True) + 1e-6)).detach()

        layer_stats = [{'loss_lm': float(emb_loss.item()), 'loss_ff': 0.0, 'g_pos': 1.0, 'g_neg': 1.0}]
        for block in self.blocks:
            stats = block.train_step_ff(cur_pos, cur_neg, target_ids)
            layer_stats.append(stats)
            cur_pos = stats['next_h_pos']
            cur_neg = stats['next_h_neg']
        return layer_stats

    @torch.no_grad()
    def generate(self, prompt_tokens: torch.Tensor, max_new_tokens: int = 16, temperature: float = 0.0) -> torch.Tensor:
        self.eval()
        gen = prompt_tokens.clone()
        for _ in range(max_new_tokens):
            h = self.embedding(gen)
            h = h / (h.norm(2, dim=-1, keepdim=True) + 1e-6)
            for block in self.blocks:
                h = block(h)
                h = h / (h.norm(2, dim=-1, keepdim=True) + 1e-6)
            logits = self.blocks[-1].local_head(h[:, -1, :])
            if temperature < 1e-3:
                next_tok = torch.argmax(logits, dim=-1, keepdim=True)
            else:
                probs = F.softmax(logits / temperature, dim=-1)
                next_tok = torch.multinomial(probs, num_samples=1)
            gen = torch.cat([gen, next_tok], dim=1)
        return gen

class StructuredPatternDataset(Dataset):
    def __init__(self, num_samples: int = 500, seq_len: int = 32, vocab_size: int = 500):
        self.num_samples = num_samples
        self.seq_len = seq_len
        self.vocab_size = vocab_size
        self.data = []
        for _ in range(num_samples):
            a = np.random.randint(10, 100)
            b = np.random.randint(100, 200)
            c = np.random.randint(200, 300)
            d = np.random.randint(300, 400)
            base = [a, b, c, d]
            full = (base * ((seq_len // 4) + 2))[:seq_len + 1]
            self.data.append(torch.tensor(full, dtype=torch.long))

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        full = self.data[idx]
        pos = full[:-1]
        targets = full[1:]
        neg = pos.clone()
        mask = torch.rand(self.seq_len) < 0.4
        neg[mask] = torch.randint(0, self.vocab_size, (mask.sum().item(),))
        return pos, neg, targets

## 3. Forward-Forward 学習の実行 & 局所収束性の可視化

In [ ]:
dataset = StructuredPatternDataset(num_samples=500, seq_len=32, vocab_size=500)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

model = ForwardForwardBitSSM(vocab_size=500, d_model=64, n_layers=3, d_state=16).to(device)

epochs = 10
history_lm = {l: [] for l in range(4)}
history_g_diff = {l: [] for l in range(3)}

print("🚀 Training Forward-Forward Bit-SSM...")
t0 = time.time()
for ep in range(1, epochs + 1):
    ep_lm = [0.0] * 4
    ep_g_diff = [0.0] * 3
    cnt = 0
    for pos, neg, targets in dataloader:
        pos, neg, targets = pos.to(device), neg.to(device), targets.to(device)
        stats = model.train_step(pos, neg, targets)
        ep_lm[0] += stats[0]['loss_lm']
        for l in range(3):
            ep_lm[l + 1] += stats[l + 1]['loss_lm']
            ep_g_diff[l] += (stats[l + 1]['g_pos'] - stats[l + 1]['g_neg'])
        cnt += 1
    for l in range(4):
        history_lm[l].append(ep_lm[l] / cnt)
    for l in range(3):
        history_g_diff[l].append(ep_g_diff[l] / cnt)
    print(f"Epoch [{ep}/{epochs}] | Emb: {history_lm[0][-1]:.3f} | L0: {history_lm[1][-1]:.3f} | L1: {history_lm[2][-1]:.3f} | L2: {history_lm[3][-1]:.3f}")
print(f"✅ Training finished in {time.time() - t0:.2f}s!")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(range(1, epochs + 1), history_lm[0], marker='o', label='Embedding Head')
for l in range(3):
    axes[0].plot(range(1, epochs + 1), history_lm[l+1], marker='o', label=f'SSM Block {l}')
axes[0].set_title('Local LM Cross-Entropy Loss per Layer (Zero Backprop)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True, linestyle='--', alpha=0.6)
axes[0].legend()

for l in range(3):
    axes[1].plot(range(1, epochs + 1), history_g_diff[l], marker='s', label=f'SSM Block {l}')
axes[1].set_title('Forward-Forward Contrastive Separation (G_pos - G_neg)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Goodness Delta (ΔG)')
axes[1].grid(True, linestyle='--', alpha=0.6)
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. 逆伝播フリーモデルによるテキスト自己回帰生成

In [ ]:
sample_prompt = dataset.data[0][:4].unsqueeze(0).to(device)
gen = model.generate(sample_prompt, max_new_tokens=16, temperature=0.0)

print("✨ Forward-Forward Autoregressive Generation Test:")
print(f"   Prompt Pattern:  {sample_prompt.tolist()[0]}")
print(f"   Expected Repeat: {dataset.data[0][:20].tolist()}")
print(f"   Model Output:    {gen.tolist()[0]}")